In [1]:
import boto3
import math
from sagemaker import get_execution_role
from pprint import pprint
import time

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Functions

In [2]:
def get_specs(str_instance):
    if str_instance == 'm5.large':
        int_vcpu = 2
        int_memory_gb = 8
    elif str_instance == 'm5.xlarge':
        int_vcpu = 4
        int_memory_gb = 16
    elif str_instance == 'm5.2xlarge':
        int_vcpu = 8
        int_memory_gb = 32
    elif str_instance == 'm5.4xlarge':
        int_vcpu = 16
        int_memory_gb = 64
    elif str_instance == 'm5.8xlarge':
        int_vcpu = 32
        int_memory_gb = 128
    elif str_instance == 'm5.12xlarge':
        int_vcpu = 48
        int_memory_gb = 192
    int_memory_mebibytes = math.ceil(int_memory_gb * 953.674)
    dict_output = {
        'int_vcpu': int_vcpu,
        'int_memory_gb': int_memory_gb,
        'int_memory_mebibytes': int_memory_mebibytes,
    }
    return dict_output

### Constants

In [3]:
str_image_name = 'genxii-lgd-eda'
int_iteration = 1
str_instance = 'm5.large'
dict_specs = get_specs(str_instance=str_instance)
int_vcpu = dict_specs['int_vcpu']
int_memory_gb = dict_specs['int_memory_gb']
int_memory_mebibytes = dict_specs['int_memory_mebibytes']
for key, val in dict_specs.items():
    print(f'{key}: {val}')

int_vcpu: 2
int_memory_gb: 8
int_memory_mebibytes: 7630


### Create compute environment

In [4]:
# initialize class
cls_client = boto3.client('batch')

In [5]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [6]:
# create compute environment
while True:
    try:
        str_compute_env_name = f'env-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_compute_environment(
            computeEnvironmentName=str_compute_env_name,
            type= 'Managed', 
            state= 'ENABLED',
            serviceRole = str_role,
            computeResources={
                #'type': 'SPOT',
                'type': 'EC2',
                'minvCpus': 0,
                'maxvCpus': 256, 
                'desiredvCpus': int_vcpu,
                'instanceTypes': [
                    str_instance,
                ], 
                'subnets': ['subnet-044e573651bb251a7'], 
                'securityGroupIds': ['sg-03904237048cdc335'], 
                'instanceRole': 'ecsInstanceRole',
                #'spotIamFleetRole': 'AmazonEC2SpotFleetTaggingRole',
            },
        )
        pprint(dict_response)
        break
    except:
        int_iteration += 1

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '153',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 23 Apr 2024 20:53:24 GMT',
                                      'x-amz-apigw-id': 'WsnmtF8_PHcEuqw=',
                                      'x-amzn-requestid': '379a7104-acc3-4d71-9f9f-6a925a177b8b',
                                      'x-amzn-trace-id': 'Root=1-66281fc4-728673567b986560418ab019'},
                      'HTTPStatusCode': 200,
                      'RequestId': '379a7104-acc3-4d71-9f9f-6a925a177b8b',
                      'RetryAttempts': 0},
 'computeEnvironmentArn': 'arn:aws:batch:

### Create job Queue

In [7]:
# create job queue (this is where AWS will store your jobs until an EC2 Instance is available to run them)
while True:
    try:
        str_job_queue_name = f'queue-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_job_queue(
            jobQueueName=str_job_queue_name,
            state='ENABLED',
            priority=1,
            computeEnvironmentOrder=[
                {
                    'order': 1,
                    'computeEnvironment': str_compute_env_name,
                },
            ]
        )
        # get arn
        str_job_queue_arn = dict_response['jobQueueArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '127',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 23 Apr 2024 20:53:29 GMT',
                                      'x-amz-apigw-id': 'WsnnkGO7PHcErQA=',
                                      'x-amzn-requestid': '3bbf5951-a382-45f6-87b5-933f00cb58f2',
                                      'x-amzn-trace-id': 'Root=1-66281fc9-463db8bd45fb9f8a65257ef5'},
                      'HTTPStatusCode': 200,
                      'RequestId': '3bbf5951-a382-45f6-87b5-933f00cb58f2',
                      'RetryAttempts': 0},
 'jobQueueArn': 'arn:aws:batch:us-west-2:

### Register job definition

In [8]:
# job definition
while True:
    try:
        str_job_definition = f'job-def-{str_image_name}-{int_iteration}'
        dict_response = cls_client.register_job_definition(
            type='container',
            containerProperties={
                'image': f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_image_name}:latest',
                'memory': int_memory_mebibytes,
                'vcpus': int_vcpu,
            },
            jobDefinitionName=str_job_definition,
        )
        # get arn
        str_job_def_arn = dict_response['jobDefinitionArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '161',
                                      'content-type': 'application/json',
                                      'date': 'Tue, 23 Apr 2024 20:53:29 GMT',
                                      'x-amz-apigw-id': 'WsnnlEFBvHcEjNQ=',
                                      'x-amzn-requestid': '84b7a1c0-8478-4205-bd26-0f300bd454c6',
                                      'x-amzn-trace-id': 'Root=1-66281fc9-43b14dd844888d0a0b3839a0'},
                      'HTTPStatusCode': 200,
                      'RequestId': '84b7a1c0-8478-4205-bd26-0f300bd454c6',
                      'RetryAttempts': 0},
 'jobDefinitionArn': 'arn:aws:batch:us-we

### Submit job

In [9]:
# # submit a job
# while True:
#     try:
#         str_job_name = f'job-name-{str_image_name}-{int_iteration}'
#         response = cls_client.submit_job(
#             jobDefinition=str_job_definition,
#             jobQueue=str_job_queue_name,
#             jobName=str_job_name,
#         )
#         pprint(response)
#         break
#     except:
#         time.sleep(1)

### Show arns

In [10]:
print(f'Job Queue ARN: {str_job_queue_arn}')
print(f'Job Definition ARN: {str_job_def_arn}')

Job Queue ARN: arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-lgd-eda-1
Job Definition ARN: arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-lgd-eda-1:5
